# Homework Assignment: GCN for Node Classification and Imputation


In [ ]:
!pip install leidenalg scanpy igraph phate scprep magic-impute -q
!pip install torch torch-geometric  -q
!wget https://cf.10xgenomics.com/samples/cell-exp/3.0.0/pbmc_10k_v3/pbmc_10k_v3_filtered_feature_bc_matrix.h5

In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree
from torch_geometric.data import Data
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import kneighbors_graph
import seaborn as sns
import magic
import phate
import scprep

print("PyTorch version:", torch.__version__)
print("PyTorch Geometric available:", True)
print("Setup complete!")


## Load PBMC12k Data from Scanpy

In [ ]:
adata = sc.read_10x_h5('pbmc_10k_v3_filtered_feature_bc_matrix.h5')

print(f"Dataset shape: {adata.shape}")
print(f"Number of cells: {adata.n_obs}")
print(f"Number of genes: {adata.n_vars}")
print(f"\nFirst few gene names: {adata.var_names[:5].tolist()}")

# Basic preprocessing
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
print(f"\nAfter filtering - Cells: {adata.n_obs}, Genes: {adata.n_vars}")

# Normalize and log-transform
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Select highly variable genes
sc.pp.highly_variable_genes(adata, n_top_genes=500)
adata = adata[:, adata.var.highly_variable]
print(f"After selecting highly variable genes: {adata.n_vars} genes")

# Perform PCA for downstream analysis
sc.tl.pca(adata, n_comps=50)
print("\nPCA completed with 50 components")

# Compute cell type annotations (clustering)
sc.pp.neighbors(adata, n_neighbors=10)
sc.tl.leiden(adata, resolution=0.5)
print(f"Number of cell clusters identified: {adata.obs['leiden'].nunique()}")

# Display cell type distribution
print("\nCell type distribution:")
print(adata.obs['leiden'].value_counts().sort_index())

## Building a Graph from Data

In [4]:
print("\n--- Building Graph from Single-Cell Data ---")

# TODO: Experiment with different k values for k-NN graph construction
k_neighbors = 3

# Use PCA representation for building the graph
X = adata.obsm['X_pca'][:, :30]  # Use first 30 PCs

# TODO: Build k-NN graph
print(f"Building k-NN graph with k={k_neighbors}...")
adjacency_matrix = kneighbors_graph(X, n_neighbors=k_neighbors, mode='connectivity', include_self=True) #use kneighbors_graph from sklearn

# TODO: Convert to edge index format (COO format)
adjacency_matrix = adjacency_matrix.tocoo()
edge_index = torch.tensor([adjacency_matrix.row, adjacency_matrix.col], dtype=torch.long) #Convert to PyG edge_index format

print(f"Number of edges: {edge_index.shape[1]}")
print(f"Edge index shape: {edge_index.shape}")

# Prepare node features (use highly variable genes)
node_features = torch.tensor(adata.X.toarray(), dtype=torch.float)
print(f"Node features shape: {node_features.shape}")

# TODO: Prepare labels (cell type annotations)
labels = torch.tensor(adata.obs['leiden'].astype('category').cat.codes.values, dtype=torch.long)
num_classes = labels.max().item() + 1  # Find unique cell types
print(f"Number of classes: {num_classes}")

# Create train/test/validation masks
train_idx, temp_idx = train_test_split(np.arange(len(labels)), test_size=0.4, random_state=42, stratify=labels.numpy())
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42, stratify=labels[temp_idx].numpy())

train_mask = torch.zeros(len(labels), dtype=torch.bool)
val_mask = torch.zeros(len(labels), dtype=torch.bool)
test_mask = torch.zeros(len(labels), dtype=torch.bool)

train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

print(f"\nTrain nodes: {train_mask.sum()}")
print(f"Validation nodes: {val_mask.sum()}")
print(f"Test nodes: {test_mask.sum()}")

# Create PyTorch Geometric Data object
data = Data(x=node_features, edge_index=edge_index, y=labels)
data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print("\nGraph data object created:")
print(data)


--- Building Graph from Single-Cell Data ---
Building k-NN graph with k=3...
Number of edges: 34611
Edge index shape: torch.Size([2, 34611])
Node features shape: torch.Size([11537, 500])
Number of classes: 13

Train nodes: 6922
Validation nodes: 2307
Test nodes: 2308

Graph data object created:
Data(x=[11537, 500], edge_index=[2, 34611], y=[11537], train_mask=[11537], val_mask=[11537], test_mask=[11537])


/var/folders/yn/wwd5hg1s66g4gjp_60m2465m0000gn/T/ipykernel_7666/2328880497.py:15: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  edge_index = torch.tensor([adjacency_matrix.row, adjacency_matrix.col], dtype=torch.long) #Convert to PyG edge_index format


## Build a GCN

In [5]:
# Standard GCN Convolutional layer from Kipf & Welling
class GCNConv(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super(GCNConv, self).__init__(aggr='add')  # "Add" aggregation (Step 5).
        self.lin = torch.nn.Linear(in_channels, out_channels)

    def forward(self, x, edge_index):
        # x has shape [N, in_channels]
        # edge_index has shape [2, E]

        # Step 1: Add self-loops to the adjacency matrix.
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))

        # Step 2: Linearly transform node feature matrix.
        x = self.lin(x)

        # Step 3: Compute normalization.
        row, col = edge_index
        deg = degree(col, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        # Step 4-5: Start propagating messages.
        return self.propagate(edge_index, x=x, norm=norm)

    def message(self, x_j, norm):
        # x_j has shape [E, out_channels]
        # Step 4: Normalize node features.
        return norm.view(-1, 1) * x_j

In [6]:
class GCNModel(torch.nn.Module):
    def __init__(self, num_node_features, hidden_features, output_features):
        super(GCNModel, self).__init__()
        self.conv1 = GCNConv(num_node_features, hidden_features)
        self.conv2 = GCNConv(hidden_features, hidden_features)
        self.conv3 = GCNConv(hidden_features, output_features)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)

        x = self.conv3(x, edge_index)
        return x

# Build a `BetterGCN`

In [ ]:
# TODO: Adapt the GCNConv class with the modifications suggested by the pset. Refer to https://pytorch-geometric.readthedocs.io/en/latest/notes/create_gnn.html for details on the MessagePassing class
class BetterGCNConv(MessagePassing):
    pass

In [ ]:
# TODO: Adapt the GCNModel class with BetterGCNConv
class BetterGCNModel(torch.nn.Module):
    pass

## Using BetterGCN for Node Classification

In [ ]:
# TODO: Initialize model
num_features = data.x.shape[1]
hidden_dim = 64
num_classes = len(torch.unique(data.y))

model = pass # Initialize the BetterGCNModel

print(f"\nModel architecture:")
print(model)
print(f"\nNumber of parameters: {sum(p.numel() for p in model.parameters())}")

# TODO: Experiment with different optimizers and learning rates
optimizer = pass #Initialzie an Adam optimizer
criterion = torch.nn.CrossEntropyLoss()

# TODO: Training function, do one step of forward pass, calculate loss, back propagate and return loss.
def train():
    pass
    return loss.item()

# TODO: Evaluation function, do one evaluation step. Return accuracy.
def evaluate(mask):
    model.eval()
    with torch.no_grad():
        pass
    return acc

# Training loop
print("\n--- Training BetterGCN for Node Classification ---")
num_epochs = 200
train_losses = []
val_accs = []
test_accs = []

for epoch in range(1, num_epochs + 1):
    loss = train()
    train_losses.append(loss)

    if epoch % 10 == 0:
        train_acc = evaluate(data.train_mask)
        val_acc = evaluate(data.val_mask)
        test_acc = evaluate(data.test_mask)
        val_accs.append(val_acc)
        test_accs.append(test_acc)

        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, '
              f'Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}')

# Final evaluation
final_test_acc = evaluate(data.test_mask)
print(f"\n{'='*60}")
print(f"Final Test Accuracy: {final_test_acc:.4f}")
print(f"{'='*60}")

# Visualize training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss')
ax1.set_title('Training Loss Over Time')
ax1.grid(True)

epochs_recorded = list(range(10, num_epochs + 1, 10))
ax2.plot(epochs_recorded, val_accs, label='Validation Accuracy', marker='o')
ax2.plot(epochs_recorded, test_accs, label='Test Accuracy', marker='s')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation and Test Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# Compute and visualize confusion matrix
model.eval()
with torch.no_grad():
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)

from sklearn.metrics import confusion_matrix, classification_report

y_true = data.y[data.test_mask].numpy()
y_pred = pred[data.test_mask].numpy()

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Cell Type Classification')
plt.show()

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

## Using MAGIC for Gene Denoising

In [ ]:
# TODO: Initilizae magic and apply it on adata.X
# MAGIC operator has many parameters. See how each of these affects the output.
# You can use this repo for reference: https://github.com/KrishnaswamyLab/MAGIC/blob/master/python/magic/magic.py
"""Set the parameters on this estimator.
        knn : int, optional, default: 5
            number of nearest neighbors on which to build kernel
        t : int, optional, default: 3
            power to which the diffusion operator is powered.
            This sets the level of diffusion. If 'auto', t is selected
            according to the R squared of the diffused data

        n_pca : int, optional, default: 100
            Number of principal components to use for calculating
            neighborhoods. For extremely large datasets, using
            n_pca < 20 allows neighborhoods to be calculated in
            roughly log(n_samples) time."""
magic_operator = pass

X_denoised = magic_operator.fit_transform(adata.X)


In [ ]:
# TODO: Initilizae phate for both the true values and predicted values.
# Similar to MAGIC, PHATE also has many parameters. Change the values and see how each one of them affects the output.

phate_data = pass
phate_denoised = pass
data_2d = phate_data.fit_transform(adata.X)
denoised_2d = phate_denoised.fit_transform(X_denoised)


In [ ]:
scprep.plot.scatter2d(data_2d, c=adata.obs['leiden'].values,
                      title='PHATE: Original Data',
                      legend_anchor=(1, 1),
                      cmap='tab10',
                      ticks=False,
                      label_prefix='Cluster ')

In [ ]:
scprep.plot.scatter2d(denoised_2d, c=adata.obs['leiden'].values,
                      title='PHATE: MAGIC Denoised',
                        legend_anchor=(1, 1),
                          cmap='tab10',
                          ticks=False,
                          label_prefix='Cluster ')